In [2]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool
from mediumterm import MediumTermMemory

In [3]:
load_dotenv()

client = Anthropic()

In [7]:
system_prompt = """You are a poker agent playing 3-handed Texas Hold'em No-Limit. 
Blinds are 1000/2000. You are Player 1 (Hero). Player 2 is P2, and Player 3 is P3. 
Think about pot odds, position, and hand strength before deciding. Make sure to carefully read and use the information in the provided json.
If you are unsure of your hand strength, use the get_equity_strength tool to calculate your win probability.
Use the take_action tool to submit your decision."""

@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. Rate how aggressive each opponent's actions are on a 1-10 scale, but only if you made it to the flop or later. 
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

@beta_tool
def get_equity_strength(hole_cards: str, board_cards: str, active_players: int) -> str:
    """Calculates the mathematical win probability (equity) of your current hand using Monte Carlo simulations.
    Call this tool when you are facing a difficult decision, a large bet, or are unsure of your exact hand strength.
    
    Args:
        hole_cards: Your two hole cards as a string (e.g., 'AsKc').
        board_cards: The visible board cards as a string (e.g., 'Td9h2c'). Leave empty '' for preflop.
        active_players: Total number of players dealt into the simulation.
    """
    
    print(f"\n   [🔍 TOOL TRIGGERED] Calculating equity for {hole_cards} | Board: {board_cards} | Players: {active_players}")
    
    # Use board_cards from the parameters
    board_input = tuple(Card.parse(board_cards)) if board_cards else ()
    
    # Use active_players and hole_cards from the parameters
    calculated_equity = calculate_hand_strength(
        active_players,      
        parse_range(hole_cards),
        board_input,
        active_players,      
        5,                
        Deck.STANDARD,
        (StandardHighHand,),
        sample_count=500
    )
    
    # Clean up the percentage for the LLM
    win_percentage = round(calculated_equity * 100, 1)
    
    return json.dumps({
        "status": "success", 
        "equity": calculated_equity,
        "message": f"You have a {win_percentage}% chance of winning."
    })

def run_turn(state_json):
    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:
    
    ```json
    {state_json}
    ```
    
    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[take_action, get_equity_strength],
        system=system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"

        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'
    
def get_visible_operations(state):
    visible = []
    showdown = sum(1 for p in state.statuses if p) > 1
    for op in state.operations:
        if isinstance(op, ChipsPushing):
            winners = {i for i, amt in enumerate(op.amounts) if amt > 0}

    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            continue
        elif isinstance(op, HoleCardsShowingOrMucking):
            if showdown or op.player_index in winners:
                visible.append(op)
        else:
            visible.append(op)
    return visible

judge_system_prompt = """You are an evaluating agent for players playing Texas Hold-em. Your job is to evaluate and provide useful information about the other non-agent players to aid Player 1 in defeating them. Do not evaluate Player 1 (Hero), focus on Player 2 (P2), and Player 3 (P3). Remember that index 0 is the agent, index 1 is Player 2 and so on. Use the generate_observation tool to submit your decision."""

@beta_tool
def generate_observation(evaluation: str) -> str:
    """Evaluate the action history of the game in the previous poker hand. 

    Args:
        evaluation: The evaluation of how the other players played. Be specific as to any mistakes or good moves each non-Hero player made. 
    """
    return json.dumps({"status": "accepted", "evaluation": evaluation})

    
def run_observation_generator(action_history):
    eval_json = {'action_history': action_history}
    messages = [{
        "role": "user",
        "content": f"""Here is the action history of the game. 
    
    ```json
    {eval_json}
    ```
    
    Generate your observations."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[generate_observation],
        system=judge_system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        for block in message.content:
            if block.type == "tool_use" and block.name == "generate_observation":
                agent_action = block.input["evaluation"]

    return agent_action


In [8]:
# Full game
med = MediumTermMemory()
med.new_game(game_id = '0', players = ['Hero, P2, P3'])
# Initial Setup
player_stacks = [10_000, 10_000, 10_000] # Starting stacks for 3 players [cite: 85]
hand_count = 0

# The Global Game Loop: Runs until only one player has chips
while len([s for s in player_stacks if s > 0]) > 1:
    reasonings = []
    active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]

    # If only one player is left, the game is over
    if len(active_indices) < 2:
        break

# Filter stacks for the engine
    current_stacks = tuple(player_stacks[i] for i in active_indices)
    hand_count += 1
    print(f"\n--- STARTING HAND {hand_count} ---")
    
    state = NoLimitTexasHoldem.create_state(
        (
            Automation.ANTE_POSTING,
            Automation.BET_COLLECTION,
            Automation.BLIND_OR_STRADDLE_POSTING,
            Automation.CARD_BURNING,
            Automation.HOLE_DEALING,
            Automation.BOARD_DEALING,
            Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
            Automation.HAND_KILLING,
            Automation.CHIPS_PUSHING,
            Automation.CHIPS_PULLING,
        ),
        True, 0, (1000, 2000), 2000, tuple(current_stacks), len(active_indices)
    )
    #big blind ante, ante, blinds, min bet, stacks, players

    # Internal Decision Loop (Per-Turn) [cite: 19, 49]
    while state.status:
        if state.can_deal_hole():
            state.deal_hole()
        elif state.can_deal_board():
            state.deal_board()
            
        elif state.actor_index is not None:
            res = get_visible_operations(state)
            obs = {
                "your_index": 0,
                "pot": state.total_pot_amount,
                "board": state.board_cards,
                "hole": state.hole_cards[state.actor_index],
                "stacks": state.stacks,
                "bets": state.bets,
                "street": street_converter(state.street_index),
                "can_fold?": state.can_fold(),
                "can_check_or_call?": state.can_check_or_call(),
                "can_raise?": state.can_complete_bet_or_raise_to(),
                "min_raise": state.min_completion_betting_or_raising_to_amount,
                "max_raise": state.max_completion_betting_or_raising_to_amount,
                "how_much_to_call": state.checking_or_calling_amount,          # how much a call costs
                "action_history": res
            }
            if state.actor_index == 0: # Agent
                # print(obs)
                action = run_turn(obs)
                if action[0]['action'] in ['check', 'call']:
                    state.check_or_call() 
                elif action[0]['action'] == 'raise':
                    state.complete_bet_or_raise_to(action[0]['amount'])
                elif action[0]['action'] == 'fold':
                    state.fold()
                try:
                    reasonings.append([action[0]['action'], action[0]['amount'], action[0]['reasoning']])
                except:
                    reasonings.append([action[0]['action'], 0, action[0]['reasoning']])
            else: 
                action_rand = random.random()
                if action_rand < 0.1 and state.can_complete_bet_or_raise_to():
        # 10% chance to raise to 3x the big blind
                    min_raise = state.min_completion_betting_or_raising_to_amount
                    max_raise = state.max_completion_betting_or_raising_to_amount # All-in
            
            # Raise to 2x the minimum required to keep it moving
                    target_raise = min(min_raise * 2, max_raise)
                    state.complete_bet_or_raise_to(target_raise)
                elif action_rand < 0.2 and state.can_fold():
        # 10% chance to fold
                    state.fold()
                else:
                    state.check_or_call()
        else:
            break

    # Update player stacks for the next hand based on payoffs [cite: 91, 92]
    # payoffs return the net change (e.g., [-2000, 4000, -2000])
    med.ingest_hand(action_history=list(map(lambda x: str(x), get_visible_operations(state))), 
                    reasoning=reasonings, chip_changes=state.payoffs)
    eval = run_observation_generator(get_visible_operations(state))
    med.log_trend(eval)
    # print(reasonings)
    for i, global_idx in enumerate(active_indices):
        player_stacks[global_idx] += int(state.payoffs[i])
    
    print(f"Hand {hand_count} Results: {state.payoffs}")
    print(f"New Stacks: {player_stacks}")

print(f"\nGAME OVER after {hand_count} hands!")
print(f"Final Winner Stacks: {player_stacks}")


--- STARTING HAND 1 ---
Hand 1 Results: [-1000, 7000, -6000]
New Stacks: [9000, 17000, 4000]

--- STARTING HAND 2 ---
Hand 2 Results: [-1000, -4000, 5000]
New Stacks: [8000, 13000, 9000]

--- STARTING HAND 3 ---

   [🔍 TOOL TRIGGERED] Calculating equity for ThKh | Board: AdQs7h | Players: 2

   [🔍 TOOL TRIGGERED] Calculating equity for ThKh | Board: AdQs7h6d | Players: 2

   [🔍 TOOL TRIGGERED] Calculating equity for ThKh | Board: AdQs7h6dJs | Players: 2
Hand 3 Results: [6000, -4000, -2000]
New Stacks: [14000, 9000, 7000]

--- STARTING HAND 4 ---
Hand 4 Results: [-1000, 500, 500]
New Stacks: [13000, 9500, 7500]

--- STARTING HAND 5 ---

   [🔍 TOOL TRIGGERED] Calculating equity for KsQs | Board: 6h9dAh | Players: 3

   [🔍 TOOL TRIGGERED] Calculating equity for KsQs | Board: 6h9dAh4s | Players: 3
Hand 5 Results: [-7500, -7500, 15000]
New Stacks: [5500, 2000, 22500]

--- STARTING HAND 6 ---
Hand 6 Results: [2000, -2000, 0]
New Stacks: [7500, 0, 22500]

--- STARTING HAND 7 ---

   [🔍 TOOL 

KeyboardInterrupt: 